# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print some key metadata attributes
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published date: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs (using the Croissant schema).

In [ ]:
# List all record sets along with their @id, name, and description
print("Available record sets in this dataset:")
for rs in dataset.metadata.record_sets:
    print(f"@id: {rs.id}\n  name: {rs.name}\n  description: {getattr(rs, 'description', '')}")
    print("  Fields/Columns:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {getattr(field, 'data_type', '')})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as listed above.

In [ ]:
# Let's prepare a list of record set @ids for extraction
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Extract data for each record set
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set: {record_set_id} with shape {df.shape}")
        else:
            print(f"No records found for record set: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Example: Show columns from the first non-empty record set
if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No dataframes could be loaded from the record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping by key attributes using field `@id`s.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Select a record set and numeric field to analyze (by `@id`)

# Display all dataframe keys to help choose which to explore
print(f"Available loaded record sets: {list(dataframes.keys())}\n")

# Example: Pick the first available DataFrame
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Exploring record set: {record_set_id}")

    # Try to find a numeric column by inspecting data types
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try to coerce all columns to numeric and use those
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except:
                continue
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
        # Filter records where the numeric_field is greater than a threshold
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}\n")
        print(filtered_df.head())

        # Normalize the field
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, normalized_col]].head())

        # Try grouping by a likely categorical column
        categorical_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and col != numeric_field]
        if categorical_candidates:
            group_field = categorical_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (showing group means):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found in this record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize numeric data distributions or field relationships using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0:
    # Use record_set_id, df, and numeric_field from previous EDA step
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()
else:
    print("No data available for plotting.")

## 6. Conclusion
We have loaded, explored, and visualized the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using its Croissant schema and the `mlcroissant` library. By referencing the dataset structure via each entity's `@id`, we ensured precise and FAIR data handling. Continue exploring the field and record set `@id`s to conduct richer domain-specific analysis and machine learning.